# Feature Extraction - Full Integration

✅ **Single notebook - all functions inline** (no separate .py file)
- Load: `data/preprocessed/preprocessed_data.csv`
- Extract 16 optimized features
- Save: back to same file (augmented with features)

## Features (16 total):
- **Text (6)**: word_count, exclamation, question, uppercase_words, uppercase_ratio, punctuation_ratio
- **Engagement (4)**: total_engagement, like_ratio, comment_ratio, like_per_char  
- **Time (6)**: hour_sin, hour_cos, day_sin, day_cos, **month_sin, month_cos** ⭐ NEW


In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set pandas options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

# =============================================================================
# TEXT-BASED FEATURES
# =============================================================================

def count_words(text: str) -> int:
    """Đếm số từ"""
    if pd.isna(text):
        return 0
    return len(text.split())


def count_exclamation(text: str) -> int:
    """Đếm số dấu chấm than - emotional signal ⭐"""
    if pd.isna(text):
        return 0
    return text.count('!')


def count_question(text: str) -> int:
    """Đếm số dấu hỏi - engagement signal ⭐"""
    if pd.isna(text):
        return 0
    return text.count('?')


def count_uppercase_words(text: str) -> int:
    """Đếm số từ viết hoa hoàn toàn - shouting ⭐"""
    if pd.isna(text):
        return 0
    words = text.split()
    return sum(1 for w in words if w.isupper() and len(w) > 1)


def calc_uppercase_ratio(text: str) -> float:
    """Tỷ lệ ký tự viết hoa - emphasis signal ⭐"""
    if pd.isna(text) or len(text) == 0:
        return 0.0
    letters = [c for c in text if c.isalpha()]
    if len(letters) == 0:
        return 0.0
    upper_count = sum(1 for c in letters if c.isupper())
    return upper_count / len(letters)


def calc_punctuation_ratio(text: str) -> float:
    """Tỷ lệ dấu câu trên tổng ký tự"""
    if pd.isna(text) or len(text) == 0:
        return 0.0
    punctuation = set('.,;:!?-()[]{}"\'/\\')
    chars = [c for c in text if not c.isspace()]
    if len(chars) == 0:
        return 0.0
    return sum(1 for c in chars if c in punctuation) / len(chars)


# =============================================================================
# ENGAGEMENT FEATURES
# =============================================================================

def calc_engagement_total(num_like: int, num_cmt: int, num_share: int) -> int:
    """Tổng engagement = like + comment + share"""
    return int(num_like + num_cmt + num_share)


def calc_like_ratio(num_like: int, total_engagement: int) -> float:
    """Tỷ lệ like / total engagement - engagement pattern ⭐"""
    if total_engagement == 0:
        return 0.0
    return num_like / total_engagement


def calc_comment_ratio(num_cmt: int, total_engagement: int) -> float:
    """Tỷ lệ comment / total engagement - engagement pattern ⭐"""
    if total_engagement == 0:
        return 0.0
    return num_cmt / total_engagement


def calc_like_per_char(num_like: int, num_char: int) -> float:
    """Like trên mỗi ký tự - normalized engagement"""
    if num_char == 0:
        return 0.0
    return num_like / num_char


# =============================================================================
# TIME-BASED FEATURES
# =============================================================================

def calc_hour_sin(hour: int) -> float:
    """Sin of hour (cyclical encoding)"""
    return np.sin(2 * np.pi * hour / 24)


def calc_hour_cos(hour: int) -> float:
    """Cos of hour (cyclical encoding)"""
    return np.cos(2 * np.pi * hour / 24)


def calc_day_sin(weekday: int) -> float:
    """Sin of weekday (cyclical encoding)"""
    return np.sin(2 * np.pi * weekday / 7)


def calc_day_cos(weekday: int) -> float:
    """Cos of weekday (cyclical encoding)"""
    return np.cos(2 * np.pi * weekday / 7)


def calc_month_sin(month: int) -> float:
    """Sin of month (cyclical encoding) - NEW ⭐"""
    return np.sin(2 * np.pi * month / 12)


def calc_month_cos(month: int) -> float:
    """Cos of month (cyclical encoding) - NEW ⭐"""
    return np.cos(2 * np.pi * month / 12)


print("✅ Imported all feature functions (including new month sin/cos)")


✅ Imported all feature functions (including new month sin/cos)


## 1. Load Preprocessed Data

In [8]:
# Load preprocessed data
preprocessed_path = '../../data/preprocessed/preprocessed_data.csv'
df = pd.read_csv(preprocessed_path)

print(f"✅ Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nShape: {df.shape}")
print(f"\nFirst row (text_strict):\n{df['text_strict'].iloc[0][:200]}...")


✅ Loaded 4736 rows, 28 columns

Columns: ['id', 'user_name', 'timestamp_post', 'post_message', 'label', 'num_char', 'num_emoji', 'num_url', 'num_hashtag', 'num_post', 'num_real', 'num_fake', 'post_ratio', 'num_like', 'num_cmt', 'num_share', 'pixel', 'image', 'num_image', 'hour', 'weekday', 'day', 'month', 'year', 'time', 'min', 'text_strict', 'text_loose']

Shape: (4736, 28)

First row (text_strict):
thăng cấp_bậc hàm đối_với cán_bộ chiến_sỹ hy_sinh ở đà_nẵng ngày đại_tướng tô_lâm bộ_trưởng bộ công_an đã ký quyết_định số qđbcax thăng cấp_bậc hàm từ đại_úy lên thiếu_tá đối_với đồng_chí đặng_thanh_t...


## 2. Import Optimized Feature Functions

In [9]:
# Create comprehensive feature extraction function
def create_all_features(df, text_col='post_message'):
    """
    Extract all optimized features from dataframe
    
    Features (16 total):
    - Text (6): word_count, exclamation, question, uppercase_words, uppercase_ratio, punctuation_ratio
    - Engagement (4): total_engagement, like_ratio, comment_ratio, like_per_char
    - Time (6): hour_sin, hour_cos, day_sin, day_cos, month_sin, month_cos (from existing columns)
    - Original columns preserved
    """
    result_df = df.copy()
    
    # TEXT FEATURES
    result_df['feat_word_count'] = result_df[text_col].apply(count_words)
    result_df['feat_exclamation'] = result_df[text_col].apply(count_exclamation)
    result_df['feat_question'] = result_df[text_col].apply(count_question)
    result_df['feat_uppercase_words'] = result_df[text_col].apply(count_uppercase_words)
    result_df['feat_uppercase_ratio'] = result_df[text_col].apply(calc_uppercase_ratio)
    result_df['feat_punctuation_ratio'] = result_df[text_col].apply(calc_punctuation_ratio)
    
    # ENGAGEMENT FEATURES
    result_df['feat_engagement_total'] = result_df.apply(
        lambda row: calc_engagement_total(row.get('num_like', 0), row.get('num_cmt', 0), row.get('num_share', 0)), 
        axis=1
    )
    result_df['feat_like_ratio'] = result_df.apply(
        lambda row: calc_like_ratio(row.get('num_like', 0), row['feat_engagement_total']), 
        axis=1
    )
    result_df['feat_comment_ratio'] = result_df.apply(
        lambda row: calc_comment_ratio(row.get('num_cmt', 0), row['feat_engagement_total']), 
        axis=1
    )
    result_df['feat_like_per_char'] = result_df.apply(
        lambda row: calc_like_per_char(row.get('num_like', 0), len(row[text_col]) if not pd.isna(row[text_col]) else 1), 
        axis=1
    )
    
    # TIME FEATURES (cyclical encoding) - from existing hour, weekday, month columns
    result_df['feat_hour_sin'] = result_df['hour'].apply(calc_hour_sin)
    result_df['feat_hour_cos'] = result_df['hour'].apply(calc_hour_cos)
    result_df['feat_day_sin'] = result_df['weekday'].apply(calc_day_sin)
    result_df['feat_day_cos'] = result_df['weekday'].apply(calc_day_cos)
    result_df['feat_month_sin'] = result_df['month'].apply(calc_month_sin)
    result_df['feat_month_cos'] = result_df['month'].apply(calc_month_cos)
    
    return result_df


def get_feature_names():
    """Get list of all feature names"""
    text_feats = ['feat_word_count', 'feat_exclamation', 'feat_question', 
                  'feat_uppercase_words', 'feat_uppercase_ratio', 'feat_punctuation_ratio']
    engagement_feats = ['feat_engagement_total', 'feat_like_ratio', 
                        'feat_comment_ratio', 'feat_like_per_char']
    time_feats = ['feat_hour_sin', 'feat_hour_cos', 'feat_day_sin', 'feat_day_cos', 
                  'feat_month_sin', 'feat_month_cos']
    
    return text_feats + engagement_feats + time_feats


print("✅ Created comprehensive feature extraction functions")
print(f"\nFeature Set:")
feature_names = get_feature_names()
print(f"Total: {len(feature_names)} features")
print(f"\nFeature list:")
for i, feat in enumerate(feature_names, 1):
    print(f"  {i:2d}. {feat}")


✅ Created comprehensive feature extraction functions

Feature Set:
Total: 16 features

Feature list:
   1. feat_word_count
   2. feat_exclamation
   3. feat_question
   4. feat_uppercase_words
   5. feat_uppercase_ratio
   6. feat_punctuation_ratio
   7. feat_engagement_total
   8. feat_like_ratio
   9. feat_comment_ratio
  10. feat_like_per_char
  11. feat_hour_sin
  12. feat_hour_cos
  13. feat_day_sin
  14. feat_day_cos
  15. feat_month_sin
  16. feat_month_cos


## 3. Extract All Features

In [10]:
print("="*60)
print("FEATURE EXTRACTION IN PROGRESS")
print("="*60)

# Extract features from text_strict (aggressive cleaning)
df_with_features = create_all_features(df, text_col='post_message')

print(f"\n✅ Feature extraction complete!")
print(f"Output shape: {df_with_features.shape}")
print(f"\nNew columns added: {len([c for c in df_with_features.columns if c.startswith('feat_')])}")


FEATURE EXTRACTION IN PROGRESS

✅ Feature extraction complete!
Output shape: (4736, 44)

New columns added: 16


## 4. Feature Analysis

In [11]:
# Get feature columns
feature_cols = [col for col in df_with_features.columns if col.startswith('feat_')]

print(f"\n{'='*60}")
print(f"FEATURE STATISTICS - {len(feature_cols)} Features")
print(f"{'='*60}\n")

# Categorize features
text_feats = [f for f in feature_cols if any(x in f for x in ['word_count', 'exclamation', 'question', 'uppercase', 'punctuation'])]
engagement_feats = [f for f in feature_cols if 'engagement' in f or 'like' in f or 'comment' in f or 'per_char' in f]
time_feats = [f for f in feature_cols if 'sin' in f or 'cos' in f]

print(f"Text Features: {len(text_feats)} | Engagement: {len(engagement_feats)} | Time: {len(time_feats)}")
print(f"\nTime features include: hour, day, MONTH (new!) sin/cos encodings")

# Show sample values
print(f"\n{'-'*60}")
print("Sample Feature Values (first 3 rows):")
print(f"{'-'*60}")
print(df_with_features[feature_cols].head(3).to_string())

# Check for null values
print(f"\n{'-'*60}")
print("Missing Values:")
print(f"{'-'*60}")
null_counts = df_with_features[feature_cols].isnull().sum()
if null_counts.sum() > 0:
    print(null_counts[null_counts > 0])
else:
    print("✅ No missing values!")

# Basic statistics
print(f"\n{'-'*60}")
print("Feature Statistics:")
print(f"{'-'*60}")
print(df_with_features[feature_cols].describe().to_string())



FEATURE STATISTICS - 16 Features

Text Features: 6 | Engagement: 4 | Time: 6

Time features include: hour, day, MONTH (new!) sin/cos encodings

------------------------------------------------------------
Sample Feature Values (first 3 rows):
------------------------------------------------------------
   feat_word_count  feat_exclamation  feat_question  feat_uppercase_words  feat_uppercase_ratio  feat_punctuation_ratio  feat_engagement_total  feat_like_ratio  feat_comment_ratio  feat_like_per_char  feat_hour_sin  feat_hour_cos  feat_day_sin  feat_day_cos  feat_month_sin  feat_month_cos
0              177                 0              0                    13              0.158520                0.035834                  20028         0.972489            0.018874           18.219832       0.707107       0.707107 -2.449294e-16       1.00000    8.660254e-01       -0.500000
1                3                 0              0                     0              0.000000                0.00

## 5. Save Extracted Features

In [12]:
# Save dataset with extracted features - save back to same location
output_path = '../../data/preprocessed/preprocessed_data.csv'
df_with_features.to_csv(output_path, index=False)

print(f"✅ Saved to: {output_path}")
print(f"   Shape: {df_with_features.shape}")
print(f"   Original columns: {len(df.columns)}")
print(f"   New columns: +{len(feature_cols)} features")
print(f"   Total columns: {len(df_with_features.columns)}")
print(f"\n🚀 Dataset ready for s3_encoding phase!")


✅ Saved to: ../../data/preprocessed/preprocessed_data.csv
   Shape: (4736, 44)
   Original columns: 28
   New columns: +16 features
   Total columns: 44

🚀 Dataset ready for s3_encoding phase!
